In [19]:
import pandas as pd
import numpy as np
import glob
import os   

In [20]:
path = "/Users/iaguirre/Library/CloudStorage/Dropbox/uca_code/camels-mx/quality_control/0_data/sites.csv"

In [21]:
sites_df = pd.read_csv(path)
sites_df["site_id_expanded"] = (
    sites_df["site_id"].str.replace("MX_", "", regex=False)
    .str.zfill(5)
    .radd("MX_")
)
sites_df

,Estacion,site_id,longitude,latitude,site_id_expanded
0,BOQUILLA_NUMERO_1,MX_22008,-95.9357,16.6472,MX_22008
1,BOLAÑOS,MX_12428,-103.7827,21.8315,MX_12428
2,ZANATEPEC,MX_22026,-94.3624,16.4831,MX_22026
3,CAMPOMAL,MX_12496,-105.1145,21.8301,MX_12496
4,EL_PUENTE,MX_20037,-99.4142,17.1483,MX_20037
5,EL_TOMASEÑO,MX_25043,-99.4354,24.2521,MX_25043
6,TEQUISQUIAPAN,MX_26151,-99.8923,20.5169,MX_26151
7,CANDELARIA,MX_30181,-91.0500,18.1830,MX_30181
8,SAN_CRISTOBAL_II,MX_12532,-103.4238,21.0373,MX_12532
9,LA_ESPERANZA,MX_25037,-97.9486,23.1622,MX_25037


In [22]:
print(sites_df)

                     Estacion   site_id  longitude  latitude site_id_expanded
0           BOQUILLA_NUMERO_1  MX_22008   -95.9357   16.6472         MX_22008
1                     BOLAÑOS  MX_12428  -103.7827   21.8315         MX_12428
2                   ZANATEPEC  MX_22026   -94.3624   16.4831         MX_22026
3                    CAMPOMAL  MX_12496  -105.1145   21.8301         MX_12496
4                   EL_PUENTE  MX_20037   -99.4142   17.1483         MX_20037
5                 EL_TOMASEÑO  MX_25043   -99.4354   24.2521         MX_25043
6               TEQUISQUIAPAN  MX_26151   -99.8923   20.5169         MX_26151
7                  CANDELARIA  MX_30181   -91.0500   18.1830         MX_30181
8            SAN_CRISTOBAL_II  MX_12532  -103.4238   21.0373         MX_12532
9                LA_ESPERANZA  MX_25037   -97.9486   23.1622         MX_25037
10            NARCISO_MENDOZA  MX_11028  -103.9647   24.0089         MX_11028
11                    TETLAMA  MX_18323   -99.2785   18.8342    

In [23]:
def flag_is_negative(value):
    if pd.isna(value):
        return 0
    elif value < 0:
        return 1
    else:
        return 0

def flag_is_nan(value):
    if pd.isna(value):
        return 1
    else:
        return 0

In [24]:
import pandas as pd

"""
drops — detect drops in the time‐series flow data.
"""

def detect_drops(data: pd.DataFrame,
               value_col: str = 'value', 
               threshold: float = 5.0):
    """
        Identify rows where the flow series has a drop (details: see publication).

        Parameters
        ----------
        data : pd.DataFrame - A DataFrame containing your flow data.
        value_col: str - The name of the column containing flow values (default 'value').
        threshold : Multiplicative factor for the relative drop (default 5.0).

        Returns
        -------
        data: pd.DataFrame - The same df with relative drops flagged in a separate column ('drops')
        """    

   
    df = data.copy()

    drop_pct = df[value_col].diff() / df[value_col].shift(1)

    df["drop_flag"] = (drop_pct < 1 / threshold - 1).astype(int)

    return df

    #return data

In [25]:
def detect_shifts(data, value_col="value", window_length=16,
                  low_diff_quantile=0.05, shift_count_thresh=8):

    df = data.copy()

    diff = df[value_col].diff()
    sign = np.sign(diff)

    sign[diff.abs() < df[value_col].abs().quantile(low_diff_quantile)] = 0

    def shifts_in_window(window):
        return np.sum(np.abs(np.diff(window)) == 2)

    shifts = sign.rolling(window=window_length).apply(
        shifts_in_window, raw=True
    ).fillna(0)

    df["shift_flag"] = (shifts > shift_count_thresh).astype(int)

    # Flag the entire window where shifts were detected
    flagged = df["shift_flag"].copy()

    for i in np.flatnonzero(flagged.to_numpy()):
        df.iloc[max(0, i - window_length + 1):i + 1,
                df.columns.get_loc("shift_flag")] = 1

    return df

In [26]:
def detect_rel_spikes(data, value_col="value", rel_threshold=5.0):
    df = data.copy()

    df["r_up"] = df[value_col].diff() / df[value_col].shift(1)
    df["r_down"] = -(df[value_col].diff(-1)) / df[value_col]

    df["rel_spike_flag"] = (
        ((df["r_up"] > rel_threshold - 1) &
         (df["r_down"] < 1 / rel_threshold - 1)) |
        ((df["r_up"] < 1 / rel_threshold - 1) &
         (df["r_down"] > rel_threshold - 1))
    ).astype(int)

    return df.drop(columns=["r_up", "r_down"])

def detect_abs_spikes(data, value_col="value", abs_threshold=0.99):
    df = data.copy()

    threshold = df[value_col].quantile(abs_threshold)

    df["abs_spike_flag"] = (
        ((df[value_col].diff() > threshold) &
         (df[value_col].diff().shift(-1) < -threshold)) |
        ((df[value_col].diff() < -threshold) &
         (df[value_col].diff().shift(-1) > threshold))
    ).astype(int)

    return df

In [27]:
def detect_truncated(
    data,
    value_col="value",
    truncation_window=7,
    high_flow_truncation_window=1,
    high_flow_quantile=0.99
):
    df = data.copy()

    def truncated(data, window):
        flag = (
            data[value_col]
            .rolling(window=window)
            .apply(lambda x: len(set(x)) == 1, raw=True)
            .fillna(0)
            .astype(int)
        )

        # Flag all observations in each truncated window
        flagged = flag.copy()

        for i in np.flatnonzero(flagged.to_numpy()):
            flag.iloc[max(0, i - window + 1):i + 1] = 1

        return flag

    df["truncation_flag"] = truncated(df, truncation_window)

    df["high_truncation_flag"] = (
        (df[value_col] > df[value_col].quantile(high_flow_quantile))
        & (truncated(df, high_flow_truncation_window) == 1)
    ).astype(int)

    return df

In [28]:
import numpy as np
import pandas as pd

def detect_high_std(data: pd.DataFrame,
                    value_col: str = "value",
                    n_std: int = 6) -> pd.DataFrame:
    """
    Flag flow values that exceed mean + n_std * std in log-transformed space.
    """
    logged = np.log(data[value_col] + 0.01)
    mean = logged.mean()
    std = logged.std()
    threshold = mean + n_std * std

    data['above_6std'] = (logged > threshold).astype(int)
    return data


In [29]:
def detect_high_values(data, value_col="value", threshold=5000):
    df = data.copy()

    df["high_flag"] = (df[value_col] > threshold).astype(int)

    return df

In [51]:
from scipy.stats import genextreme


def calculate_amax_old(data: pd.DataFrame,
                   value_col: str = "value") -> pd.DataFrame:
    """
    Calculate the annual maximum flow for each water year without modifying the
    input DataFrame. Water year runs from October to September.
    """
    working = data.copy()
    
    working['datetime'] = pd.to_datetime(working['datetime'])

    water_year = [dt.year if dt.month < 10 else dt.year + 1 for dt in working['datetime']]
    locator = working.groupby(water_year)[value_col].idxmax()
    annual_maximas = working.loc[locator]

    return annual_maximas

    # Water year: October to September
    water_year = working.index.year + (working.index.month >= 10)

    # Remove missing streamflow values
    valid = working.dropna(subset=[value_col])

    # Return an empty DataFrame if no valid observations exist
    if valid.empty:
        return working.iloc[0:0].copy()

    # Calculate water year for valid observations
    valid_water_year = (
        valid.index.year + (valid.index.month >= 10)
    )

    # Identify the date of the maximum streamflow in each water year
    locator = valid.groupby(valid_water_year)[value_col].idxmax()

    # Extract annual maximum observations
    annual_maximas = valid.loc[locator].copy()

    return annual_maximas


def calculate_amax(data, value_col="value"):
    working = data.copy()

    # Use datetime column or DatetimeIndex
    if "datetime" in working.columns:
        dates = pd.to_datetime(working["datetime"])
    else:
        dates = pd.Series(
            pd.to_datetime(working.index),
            index=working.index
        )

    # Water year: October to September
    water_year = dates.dt.year + (dates.dt.month >= 10)

    # Exclude missing streamflow values and dates
    valid_mask = working[value_col].notna() & dates.notna()
    valid = working.loc[valid_mask].copy()
    valid_water_year = water_year.loc[valid_mask]

    # Handle datasets without valid observations
    if valid.empty:
        return working.iloc[0:0].copy()

    # Identify annual maximum for each water year
    locator = valid.groupby(valid_water_year)[value_col].idxmax()

    # Extract annual maximum observations
    annual_maximas = valid.loc[locator].copy()

    return annual_maximas

def gev_fit(data: pd.DataFrame,
            value_col: str = "value") -> pd.DataFrame:
    """
    Fit a GEV distribution to annual maxima and compute the return period for
    each observation.
    """
    annual_maximas = calculate_amax(data, value_col=value_col)
    annual_maximas = annual_maximas.dropna(subset=[value_col])

    params = genextreme.fit(annual_maximas[value_col])
    cdf_values = genextreme.cdf(data[value_col], *params)
    data['return_period'] = 1 / (1 - cdf_values)

    return data


def detect_high_return_period(data: pd.DataFrame,
                              threshold_rp: int = 1000) -> pd.DataFrame:
    """
    Flag timesteps whose estimated return period exceeds threshold_rp.
    """
    data['above_1000_rp'] = (data['return_period'] > threshold_rp).astype(int)
    return data



def detect_above_second_highest_amax(data: pd.DataFrame,
                                     value_col: str = "value",
                                     factor_second_highest: int = 2) -> pd.DataFrame:
    """
    Flag the top annual maximum when it is larger than the second highest annual
    maximum by the requested factor.
    """
    amax = calculate_amax(data, value_col=value_col)
    amax = amax.dropna(subset=[value_col]).sort_values(value_col, ascending=False)
    amax = amax[amax[value_col] != 0]

    data['top_amax'] = 0

    if len(amax) > 1:
        if (amax.iloc[0][value_col] / amax.iloc[1][value_col]) > factor_second_highest:
            top_datetime = amax.iloc[0]['datetime']
            data.loc[pd.to_datetime(data['datetime']) == pd.to_datetime(top_datetime), 'top_amax'] = 1

    return data



In [44]:
flag_columns = [
    "negative_flag",
    "nan_flag",
    "drop_flag",
    "shift_flag",
    "rel_spike_flag",
    "abs_spike_flag",
    "truncation_flag",
    "high_truncation_flag",
    "high_flag",
    "above_6std",
    "above_1000_rp",
    "top_amax",
    "qc",
    "hf_qc"
]



# Completeness flag

Flag	n+	Completitud aproximada
0	0–50	86–100%
1	51–100	73–86%
2	101–150	59–72%
3	151–200	45–59%
4	201–250	32–45%
5	251–300	18–31%
6	301–350	4–18%
7	351–366	0–4%

In [45]:
flag_columns = [
    "negative_flag",
    "nan_flag",
    "drop_flag",
    "shift_flag",
    "rel_spike_flag",
    "abs_spike_flag",
    "truncation_flag",
    "high_truncation_flag",
    "high_flag",
    "completeness_flag"
]

Code	Meaning
0	No issues found
1	Negative value
2	Relative spike
3	Absolute spike
4	Drop
5	Fluctuation
6	Truncated low flows/levels
7	Truncated high flows/levels
8	Any combination of two or more QC issues
9	NaN

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def create_streamflow_netcdf(df, output_file):

    data = df.copy()

    # Ensure datetime is the index
    if "datetime" in data.columns:
        data["datetime"] = pd.to_datetime(data["datetime"])
        data = data.set_index("datetime")

    data = data.sort_index()
    data.index.name = "time"

    # Create NetCDF dataset
    ds = xr.Dataset(
        data_vars={
            "streamflow": (
                "time",
                data["value"].to_numpy(dtype=np.float32)
            ),
            "streamflow_filtered": (
                "time",
            data["value"].where(data["qc"] == 0).to_numpy(dtype=np.float32)
        ),
            "level": (
                "time",
                data["level"].to_numpy(dtype=np.float32)
            ),
            "qcflag": (
                "time",
                data["qc"].fillna(-9999).to_numpy(dtype=np.int16)
            ),
            "completeness_flag": (
                "time",
                data["completeness_flag"].fillna(-9999).to_numpy(dtype=np.int16)
            ),
            "highflow_qc": (
                "time",
                data["hf_qc"].fillna(-9999).to_numpy(dtype=np.int16)
            )
        },
        coords={
            "time": data.index
        }
    )

    # Streamflow metadata
    ds["streamflow"].attrs = {
        "long_name": "Daily streamflow",
        "units": "m3 s-1",
        "description": (
            "Daily streamflow observations. "
            "Quality-control information is provided separately "
            "in qcflag and highflow_qc."
        )
    }

    # Basic QC metadata
    ds["qcflag"].attrs = {
        "long_name": "Basic streamflow quality control",
        "description": (
            "0: No issues found; "
            "1: Negative value; "
            "2: Relative spike; "
            "3: Absolute spike; "
            "4: Drop; "
            "5: Fluctuation; "
            "6: Truncated low flows/levels; "
            "7: Truncated high flows/levels; "
            "8: Any combination of two or more QC issues; "
            "9: NaN."
        )
    }

    # Completeness metadata
    ds["completeness_flag"].attrs = {
        "long_name": "Annual streamflow completeness classification",
        "description": (
            "Number of missing daily streamflow observations "
            "within each calendar year. "
            "0: 0-50 missing days; "
            "1: 51-100; "
            "2: 101-150; "
            "3: 151-200; "
            "4: 201-250; "
            "5: 251-300; "
            "6: 301-350; "
            "7: 351-366. "
            "The same flag is assigned to every observation "
            "within a calendar year."
        )
    }

    # High-flow QC metadata
    ds["highflow_qc"].attrs = {
        "long_name": "High-flow quality control",
        "description": (
            "High-flow quality-control classification. "
            "0: Not a high flow; "
            "1: Unrealistically high event; "
            "2: Exceptionally high event. "
            "Rainfall and regional hydrometric checks are excluded."
        )
    }

    # Global metadata
    ds.attrs = {
        "title": "CAMELS-MX daily streamflow",
        "summary": (
            "Daily streamflow observations with basic quality control, "
            "annual completeness classification, and high-flow "
            "quality-control information."
        ),
        "source": "CAMELS-MX",
        "frequency": "daily"
    }

    # NetCDF encoding
    encoding = {
        "streamflow": {
            "dtype": "float32",
            "_FillValue": np.float32(-9999.0),
            "zlib": True,
            "complevel": 4
        },
        "qcflag": {
            "dtype": "int16",
            "_FillValue": np.int16(-9999),
            "zlib": True,
            "complevel": 4
        },
        "completeness_flag": {
            "dtype": "int16",
            "_FillValue": np.int16(-9999),
            "zlib": True,
            "complevel": 4
        },
        "highflow_qc": {
            "dtype": "int16",
            "_FillValue": np.int16(-9999),
            "zlib": True,
            "complevel": 4
        }
    }

    ds.to_netcdf(
        output_file,
        engine="netcdf4",
        encoding=encoding
    )

    return ds

In [57]:
folder = "/Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data"
tests = "/Users/iaguirre/Documents/camels_mx_wip/camels_mx_v0"

os.makedirs(tests, exist_ok=True)

for site in sites_df["site_id_expanded"]:
    station_id = site.replace("MX_", "").zfill(5)
    file = os.path.join(folder, f"B{station_id}.csv")
    print(f"Processing file for site {site}: {file}")

    try:
        #df = pd.read_csv(file, parse_dates=["datetime"], index_col="datetime")
        #df = pd.read_csv(file, skiprows=0, parse_dates=["datetime"], index_col="datetime")
        df = pd.read_csv(
            file,
            skiprows=8,
            parse_dates=True,
            index_col=[0],
            names=["datetime", "level", "value"],
            dtype={"level": float, "value": float},
            na_values=["-"]
        )
        #df = df.rename(columns={"discharge": "value"})
        
        df["resolution"] = 60 * 24

        # QC flags

        df["negative_flag"] = df["value"].map(flag_is_negative)
        df["nan_flag"] = df["value"].map(flag_is_nan)
        df = detect_drops(df, value_col="value", threshold=5.0)
        df = detect_shifts(df, value_col="value")
        df = detect_rel_spikes(df, value_col="value")
        df = detect_abs_spikes(df, value_col="value")
        df = detect_truncated(
            df,
            value_col="value",
            truncation_window=7,
            high_flow_truncation_window=1,
            high_flow_quantile=0.99
        )


        qc_mapping = {
            "negative_flag": 1,
            "rel_spike_flag": 2,
            "abs_spike_flag": 3,
            "drop_flag": 4,
            "shift_flag": 5,
            "truncation_flag": 6,
            "high_truncation_flag": 7
        }

        flags = df[list(qc_mapping)].eq(1)

        df["qc"] = 0

        for col, code in qc_mapping.items():
            df.loc[flags[col], "qc"] = code

        # Any combination of two or more QC issues
        df.loc[flags.sum(axis=1) > 1, "qc"] = 8

        # Missing values
        df.loc[df["nan_flag"] == 1, "qc"] = 9


        # Completeness flags

        df["year"] = df.index.year

        df["yearly_nans"] = df.groupby("year")["value"].transform(
            lambda x: x.isna().sum()
        )

        df["completeness_flag"] = np.maximum(
            0, np.ceil((df["yearly_nans"] - 50) / 50)
        ).astype(int)

        # High flow flags

        df = df.reset_index()
        df = detect_high_values(df, value_col="value", threshold=5000)
        df = detect_high_std(df, value_col="value", n_std=6)
        df = gev_fit(df)
        df = detect_high_return_period(df)
        df = detect_above_second_highest_amax(df)

        highflow_columns = [
                "above_6std",
                "above_1000_rp",
                "top_amax"
            ]

        df["hf_qc"] = 0

        df.loc[df["high_flag"] == 1, "hf_qc"] = 1

        df.loc[
            (df["above_6std"] == 1) |
            (df["above_1000_rp"] == 1),
            "hf_qc"
        ] = 2

        clean_data = df.copy()
        clean_data.loc[clean_data["qc"] != 0, "value"] = np.nan

        data = df[["datetime", "value", "qc", "hf_qc"]]
        data = data.set_index("datetime")
        data["qc_flags"] = data["qc"].astype(str) + data["hf_qc"].astype(str)


        df = df.set_index("datetime")
        df.to_csv(os.path.join(tests, "df", f"{site}.csv"))
        data.to_csv(os.path.join(tests, "data", f"{site}_data.csv"))

        ds = create_streamflow_netcdf(
            df,
            os.path.join(tests, "netcdf", f"{site}.nc"))
        

    except Exception as e:
        print(f"Error processing site {site}: {e}")
        import traceback
        traceback.print_exc()

Processing file for site MX_22008: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B22008.csv
Processing file for site MX_12428: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B12428.csv
Processing file for site MX_22026: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B22026.csv
Processing file for site MX_12496: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B12496.csv
Processing file for site MX_20037: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B20037.csv
Processing file for site MX_25043: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B25043.csv
Processing file for site MX_26151: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B26151.csv
Processing file for site MX_30181: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B30181.csv
Processing file for site MX_12532: /Users/iaguirre/Documents/camels_mx_wip/cenagua_dataset/data/B12532.csv
Processing file for site MX_25037: /U

/var/folders/rk/cmc28xj96vg23j8hp2qt9qt40000gn/T/ipykernel_15025/3497278463.py:87: RuntimeWarning: divide by zero encountered in divide
  data['return_period'] = 1 / (1 - cdf_values)


In [53]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31236 entries, 0 to 31235
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              31236 non-null  datetime64[us]
 1   level                 0 non-null      float64       
 2   value                 27732 non-null  float64       
 3   resolution            31236 non-null  int64         
 4   negative_flag         31236 non-null  int64         
 5   nan_flag              31236 non-null  int64         
 6   drop_flag             31236 non-null  int64         
 7   shift_flag            31236 non-null  int64         
 8   rel_spike_flag        31236 non-null  int64         
 9   abs_spike_flag        31236 non-null  int64         
 10  truncation_flag       31236 non-null  int64         
 11  high_truncation_flag  31236 non-null  int64         
 12  qc                    31236 non-null  int64         
 13  year                  31236

In [54]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 31236 entries, 1935-12-01 to 2021-06-07
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   value     27732 non-null  float64
 1   qc        31236 non-null  int64  
 2   hf_qc     31236 non-null  int64  
 3   qc_flags  31236 non-null  str    
dtypes: float64(1), int64(2), str(1)
memory usage: 1.2 MB


In [ ]:
df["completeness_flag"].describe()

In [ ]:
data[]